# Formats de stockage : CSV, Pickle, Parquet, JSON ou Feather ?

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moncoachdata/MonShotData/blob/main/Optimisation%20du%20temps%20d%27exe%CC%81cution/Formats-de-Stockage-DataFrame.ipynb)

📖 **Le shot complet** : [Tu optimises ton code, mais pas tes fichiers ? Grosse erreur.](https://www.monshotdata.com/p/optimisation-format-stockage)

---

Tu passes du temps à accélérer ton code, mais le **format** dans lequel tu enregistres tes DataFrames peut changer le temps d'écriture, le temps de lecture et la taille du fichier du simple au centuple. Ici tu mesures tout ça toi-même, sur 1 million de lignes, pour cinq formats et deux types de données.

**À la fin de ce notebook, tu sauras :**

- mesurer le temps d'écriture, le temps de lecture et la taille sur disque de CSV, Pickle, Parquet, JSON et Feather ;
- voir que le meilleur format dépend des données : des nombres seuls ou un mélange de nombres et de texte ;
- choisir un format selon ton besoin : vitesse, taille, ou compatibilité avec d'autres outils.

### Avant de lancer

- **Installation** : il faut `pyarrow` pour lire et écrire Parquet et Feather. Il est déjà installé sur Colab ; sinon `pip install pyarrow`.
- **Les données** : deux DataFrames de 1 million de lignes générés dans le notebook : 20 colonnes de nombres, puis 10 colonnes de nombres et 10 colonnes de texte (5 lettres au hasard).
- **Les fichiers** : ils sont écrits dans un **dossier temporaire**, supprimé automatiquement à la fin : rien ne reste sur ton disque.
- **Durée et mémoire** : environ 3 minutes et moins de 2 Go de mémoire. Les durées dépendent de ta machine et des versions de Pandas et PyArrow : compare les formats entre eux plutôt que les valeurs absolues.

🧪 *Testé le 22 novembre 2026 : exécution complète de haut en bas (Python 3.13, pandas 3.0, NumPy 2.5 et PyArrow 26). Les sorties affichées viennent de cette exécution.*

## 1. Mise en place

In [1]:
import os
import tempfile
import time

import numpy as np
import pandas as pd

# Dossier temporaire pour les fichiers de test : supprimé à la fin du notebook
dossier = tempfile.TemporaryDirectory()
print("Fichiers écrits dans un dossier temporaire")

Fichiers écrits dans un dossier temporaire


## 2. La mesure

Une fonction mesure, pour un format donné, trois choses : le **temps d'écriture**, le **temps de lecture** et la **taille du fichier**. Chaque mesure est répétée 3 fois et on garde le meilleur temps, le moins perturbé par les autres tâches de la machine.

Les cinq formats :

| Format | Type | Particularité |
|---|---|---|
| CSV | texte | lisible partout (Excel, tout langage), mais le type des colonnes est perdu |
| Pickle | binaire Python | conserve exactement les objets Python, mais ne se lit qu'avec Python |
| Parquet | binaire en colonnes | compressé, conserve les types, lu par Spark, DuckDB, Polars... |
| JSON | texte | lisible partout, très verbeux |
| Feather | binaire en colonnes | très rapide, fait pour échanger des données entre Python et R |

In [2]:
# format -> (extension, fonction d'écriture, fonction de lecture)
FORMATS = {
    "CSV":     ("csv",     lambda d, p: d.to_csv(p, index=False), pd.read_csv),
    "Pickle":  ("pickle",  lambda d, p: d.to_pickle(p),           pd.read_pickle),
    "Parquet": ("parquet", lambda d, p: d.to_parquet(p),          pd.read_parquet),
    "JSON":    ("json",    lambda d, p: d.to_json(p),             pd.read_json),
    "Feather": ("feather", lambda d, p: d.to_feather(p),          pd.read_feather),
}

def meilleur_temps(fonction, repetitions=3):
    """Exécute la fonction plusieurs fois et renvoie le temps le plus court."""
    temps = []
    for _ in range(repetitions):
        debut = time.perf_counter()
        fonction()
        temps.append(time.perf_counter() - debut)
    return min(temps)

def compare_formats(df, prefixe):
    lignes = []
    for nom, (ext, ecrire, lire) in FORMATS.items():
        chemin = os.path.join(dossier.name, f"{prefixe}.{ext}")
        t_ecriture = meilleur_temps(lambda: ecrire(df, chemin))
        t_lecture = meilleur_temps(lambda: lire(chemin))
        taille = os.path.getsize(chemin) / 1e6   # en Mo
        lignes.append((nom, t_ecriture, t_lecture, taille))
    return pd.DataFrame(lignes, columns=["Format", "Écriture (s)", "Lecture (s)", "Taille (Mo)"]).set_index("Format")

## 3. Cas 1 : des données numériques

20 colonnes d'entiers tirés au hasard entre 0 et 100 000, sur 1 million de lignes.

In [3]:
rng = np.random.default_rng(0)   # graine fixée pour retrouver les mêmes données

num_rows = 1_000_000
num_columns = 20

df_num = pd.DataFrame({
    f"column_{i+1}": rng.integers(0, 100_001, size=num_rows)
    for i in range(num_columns)
})

print(df_num.shape)
df_num.head(3)

(1000000, 20)


,column_1,column_2,column_3,column_4,column_5,column_6,column_7,column_8,column_9,column_10,column_11,column_12,column_13,column_14,column_15,column_16,column_17,column_18,column_19,column_20
0,85063,72185,98791,18163,35198,42191,54430,2341,9014,26551,79808,95034,47979,14458,16903,21978,54917,63608,47730,48593
1,63696,89026,25279,40487,44449,99968,15138,68975,65313,74709,59191,57496,29907,35915,24133,72254,2226,16093,32019,64679
2,51114,59328,860,95598,60628,26271,66622,98206,96231,82868,72375,7769,26799,50441,22372,88377,42623,16725,87468,23940


In [4]:
resultats_num = compare_formats(df_num, "numerique")
resultats_num.round(3)

,Écriture (s),Lecture (s),Taille (Mo)
Format,,,
CSV,3.798,1.063,117.779
Pickle,0.081,0.075,160.001
Parquet,0.390,0.078,51.380
JSON,2.508,15.222,295.557
Feather,0.248,0.078,82.928


In [5]:
def resume(resultats):
    """Indique, pour chaque critère, le format gagnant."""
    for colonne in resultats.columns:
        gagnant = resultats[colonne].idxmin()
        print(f"{colonne:13s} : {gagnant} ({resultats.loc[gagnant, colonne]:.3g})")

resume(resultats_num)

Écriture (s)  : Pickle (0.0813)
Lecture (s)   : Pickle (0.0751)
Taille (Mo)   : Parquet (51.4)


À lire dans le tableau :

- le **CSV** est le plus lent à écrire et son fichier fait plus du double de celui de Parquet : chaque nombre est écrit en texte, caractère par caractère ;
- le **JSON** est le plus lent à relire, et lui aussi très volumineux ;
- les formats binaires (**Pickle**, **Feather**, **Parquet**) écrivent et relisent les nombres presque instantanément ; **Parquet** est en plus compressé, donc son fichier est de loin le plus petit, alors que Pickle n'est pas compressé et pèse plus lourd que le CSV.

## 4. Cas 2 : des données mixtes (nombres et texte)

On garde 10 colonnes de nombres et on ajoute 10 colonnes de **texte** (chaînes de 5 lettres au hasard). Le texte est plus coûteux à stocker : le classement des formats peut changer.

In [6]:
import string

lettres = np.array(list(string.ascii_letters))

def textes_aleatoires(n, longueur=5):
    """n chaînes de `longueur` lettres tirées au hasard."""
    codes = rng.integers(0, len(lettres), size=(n, longueur))
    return np.ascontiguousarray(lettres[codes]).view(f"<U{longueur}").ravel()

donnees = {f"numeric_{i+1}": rng.integers(0, 100_001, size=num_rows) for i in range(10)}
for i in range(10):
    donnees[f"string_{i+1}"] = textes_aleatoires(num_rows)

df_mixte = pd.DataFrame(donnees)

print(df_mixte.shape)
df_mixte.head(3)

(1000000, 20)


,numeric_1,numeric_2,numeric_3,numeric_4,numeric_5,numeric_6,numeric_7,numeric_8,numeric_9,numeric_10,string_1,string_2,string_3,string_4,string_5,string_6,string_7,string_8,string_9,string_10
0,1120,52467,73652,78085,60281,56719,92606,98920,31104,89507,bDHih,pLwld,bKxSe,xbwoC,VDaUv,ipzee,aGGAD,HqUof,RcjMI,KhXft
1,31044,91740,38686,16864,26281,92342,99461,20130,53003,29309,XwCyA,qWwIe,FFmAI,tGlPu,YfLhj,nLRKW,JYzhk,kwPeR,oLvxe,VzsSC
2,80071,58277,62584,39051,61690,79675,13393,56454,33606,47037,Nshto,OJwEo,wNZXe,owvtd,yKZRF,uXEom,MVPdt,mFZmf,FOYvH,FeZMp


In [7]:
print(df_mixte.dtypes.value_counts())

int64    10
str      10
Name: count, dtype: int64


In [8]:
resultats_mixte = compare_formats(df_mixte, "mixte")
resultats_mixte.round(3)

,Écriture (s),Lecture (s),Taille (Mo)
Format,,,
CSV,4.153,3.271,118.888
Pickle,0.144,0.040,210.003
Parquet,0.735,0.180,97.687
JSON,3.180,16.448,316.665
Feather,0.298,0.133,131.701


In [9]:
resume(resultats_mixte)

Écriture (s)  : Pickle (0.144)
Lecture (s)   : Pickle (0.0403)
Taille (Mo)   : Parquet (97.7)


## 5. Comparer les deux cas

Les deux tableaux côte à côte, en divisant chaque mesure par celle du CSV (1,00 = même résultat que le CSV, 0,10 = dix fois mieux) :

In [10]:
def relatif_au_csv(resultats):
    return (resultats / resultats.loc["CSV"]).round(2)

pd.concat({"Numériques": relatif_au_csv(resultats_num),
           "Mixtes": relatif_au_csv(resultats_mixte)}, axis=1)

Numériques                               Mixtes              \
        Écriture (s) Lecture (s) Taille (Mo) Écriture (s) Lecture (s)   
Format                                                                  
CSV             1.00        1.00        1.00         1.00        1.00   
Pickle          0.02        0.07        1.36         0.03        0.01   
Parquet         0.10        0.07        0.44         0.18        0.06   
JSON            0.66       14.31        2.51         0.77        5.03   
Feather         0.07        0.07        0.70         0.07        0.04   

                     
        Taille (Mo)  
Format               
CSV            1.00  
Pickle         1.77  
Parquet        0.82  
JSON           2.66  
Feather        1.11

Ce qu'on retient :

- **Parquet** reste le meilleur compromis : fichier petit, écriture et lecture rapides dans les deux cas, types conservés, et compatible avec tout l'écosystème de la donnée ;
- **Pickle** et **Feather** sont les plus rapides, mais Pickle n'est pas portable (Python uniquement, et risqué si le fichier vient d'une source inconnue) ;
- **CSV** et **JSON** sont les plus lents et les plus lourds, mais restent utiles pour échanger un fichier lisible par n'importe quel outil ;
- avec du **texte**, Parquet gagne beaucoup moins en taille que sur des nombres : des lettres tirées au hasard se compressent mal.

## 6. Attention : tes chiffres ne seront pas identiques aux nôtres

Ces mesures dépendent de plusieurs choses que tu peux faire varier :

- **la version de Pandas** : depuis Pandas 3, les colonnes de texte ont un type dédié (`str`) beaucoup plus efficace que l'ancien type `object`. Avec Pandas 2 et des colonnes `object`, les temps du cas « texte » peuvent être nettement plus élevés ;
- **le contenu des données** : un texte qui se répète beaucoup (des noms de villes, des catégories) se compresse bien mieux que des lettres tirées au hasard, et Parquet en profite ;
- **la machine** : disque, processeur et mémoire changent les durées absolues.

Le bon réflexe : refais ce test sur **tes** données avant de choisir un format.

## 7. Nettoyage

On supprime le dossier temporaire et tous les fichiers de test.

In [11]:
dossier.cleanup()
print("Dossier supprimé :", not os.path.exists(dossier.name))

Dossier supprimé : True


## 8. Ce qu'il faut retenir

- Le format de stockage d'un DataFrame influence fortement le temps d'écriture, le temps de lecture et la taille du fichier.
- CSV et JSON sont des formats texte : universels, mais les plus lents et les plus volumineux.
- Pickle et Feather sont très rapides ; Pickle ne se lit qu'avec Python et ne doit jamais être chargé depuis une source inconnue.
- Parquet offre généralement le meilleur compromis : compressé, rapide, types conservés et lisible par la plupart des outils (Spark, DuckDB, Polars...).
- Le classement dépend des données (nombres ou texte, valeurs répétées ou non) et des versions de Pandas : mesure sur tes propres données.

---

## Pour aller plus loin

- [Le meilleur format de fichier pour stocker un DataFrame Pandas](https://www.monshotdata.com/p/meilleur-format-stockage-dataframe-pandas) — la même question, avec plus de détails sur chaque format
- [Accélérer de 5x l'I/O des fichiers Parquet de Pandas](https://www.monshotdata.com/p/accelerer-io-parquet-de-pandas) — aller plus vite encore avec Parquet
- [Tous les notebooks des shots](https://github.com/moncoachdata/MonShotData) — le code Python de la newsletter, rangé par thème